# 🚀 Rest-Mex 2025: Sentiment Analysis Expert Pipeline
**Autor:** Senior Data Scientist AI Collaborator  
**Objetivo:** Clasificación de polaridad (1-5) utilizando Bi-LSTM y Embeddings pre-entrenados.

---

## 1. Configuración y Hardware
Validamos la potencia de la RTX 3090.

In [ ]:
import torch
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

## 2. Preprocesamiento Avanzado
Limpieza quirúrgica de texto y manejo de desbalance de clases.

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    # Normalización de caracteres y encoding
    text = text.lower()
    text = "".join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^a-zñ!¡?¿ ]', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text) # Reducir repeticiones
    return text.strip()

print("Sugerencia: Carga el dataset y usa 'clean_text' en una columna concatenada (Title + Review).")

## 3. Definición de la Bi-LSTM
Arquitectura robusta con Dropout Espacial.

In [ ]:
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix=None):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        
        self.spatial_dropout = nn.Dropout(0.3) # Dropout estándar para simplificar
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.bn = nn.BatchNorm1d(hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.spatial_dropout(x)
        lstm_out, _ = self.lstm(x)
        pooled, _ = torch.max(lstm_out, dim=1)
        out = self.bn(pooled)
        return self.fc(out)

## 4. Próximos Pasos
1. Cargar el dataset `.csv`.
2. Generar el vocabulario y la matriz de embeddings (FastText).
3. Entrenar usando `CrossEntropyLoss(weight=class_weights)`.